Compute CLIP embeddings for all GML rendered images.

This is very quick, only takes like 2min per 3000 drawings

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import clip
from PIL import Image
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt
import tqdm.notebook as tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
# folder = Path("./data/gml_images")
folder = Path("./data/gml_renderings")
len(sorted(folder.iterdir()))

In [ ]:
clip.available_models()

# First test using CLIP example 0-shot classification code sample

In [ ]:
# model, preprocess = clip.load("ViT-B/32", device=device) # "Base"
# model, preprocess = clip.load("ViT-L/14", device=device) # "Large"
model, preprocess = clip.load("ViT-L/14@336px", device=device) # "Large 336"

In [ ]:
# Test
imorig = Image.open(folder / "69804.jpg")
strings = ["a diagram", "a dog", "a cat", "graffiti", "truck", "MERRY X-MAS POLO"]

image = preprocess(imorig).unsqueeze(0).to(device)
text = clip.tokenize(strings).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    
    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

# Print results
print(f'Embedding has dimension {image_features.shape}, {text_features.shape}')
print('-' * 80)
with np.printoptions(precision=3, suppress=True):
    for t, prob in zip(strings, probs[0]):
        print(f"{t:<20}: {prob:.2f}")

In [ ]:
# Display image
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(imorig)
axes[1].imshow(image[0].cpu().numpy().transpose(1, 2, 0))

# Now loop through and do it all

In [ ]:
# Create from scratch

all_features = {}
all_files = sorted(list(folder.glob('*.jpg')))
all_files = [f for f in all_files if f.name not in all_features]
# Note: batching doesn't make things that much faster (2min vs 1m30s)
for im in tqdm.tqdm(all_files):
    try:
        imorig = Image.open(im)
        image = preprocess(imorig).unsqueeze(0).to(device)
        with torch.no_grad():
            image_features = model.encode_image(image).cpu().numpy()
    except Exception as e:
        print(f"Error with {im}: {e}")
        image_features = np.zeros((1, 768))
    all_features[im.name] = image_features

np.save(folder / 'features.npy', all_features)
pickle.dump(all_features, open(folder / 'features.pkl', 'wb'))

file = folder / 'CLIP_ViT-L_14@336px.pkl'
np.save(file.with_suffix('.npy'), all_features)
pickle.dump(all_features, open(file, 'wb'))

In [ ]:
!cp {folder / 'features.pkl'} {folder / 'features.pkl.bak'}

In [ ]:
# Append

f = folder / 'CLIP_ViT-L_14@336px.pkl'

all_features = pickle.load(open(f, 'rb'))
all_files = sorted(list(folder.glob('*.jpg')))
all_files = [f for f in all_files if f.name not in all_features]
# Note: batching doesn't make things that much faster (2min vs 1m30s)
for im in tqdm.tqdm(all_files):
    try:
        imorig = Image.open(im)
        image = preprocess(imorig).unsqueeze(0).to(device)
        with torch.no_grad():
            image_features = model.encode_image(image).cpu().numpy()
    except Exception as e:
        print(f"Error with {im}: {e}")
        image_features = np.zeros((1, 768))
    all_features[im.name] = image_features

!cp {folder / 'features.pkl'} {folder / 'features.pkl.bak'}
np.save(folder / 'features.npy', all_features)
pickle.dump(all_features, open(folder / 'features.pkl', 'wb'))

# np.save(f.with_suffix('.npy'), all_features)
# pickle.dump(all_features, open(f, 'wb'))

# Check GML number alignment with dataset

In [ ]:
# Load GML dataset
from diffusion_policy_gml.dataset import GmlDatasetNoSliding

dataset_folder = Path('./data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr')
dataset = GmlDatasetNoSliding(dataset_folder, -1)

In [ ]:
# Re-load CLIP embeddings file just in case
with open(folder / 'features.pkl', 'rb') as f:
    all_features = pickle.load(f)
keys = sorted(all_features.keys(), key=lambda x: int(x.split('.')[0]))
print(keys[:10])

In [ ]:
from IPython.display import Image as Image_
display(Image_(filename=folder / keys[-1]))
plt.plot(*dataset[0]['obs'].T)

In [ ]:
# Another one
keys_reversed = keys[::-1]

i = 30
display(Image_(filename=folder / keys_reversed[i]))
plt.plot(*dataset[i]['obs'].T)

In [ ]:
len(keys_reversed)

In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(20, 5))
for i in range(10):
    # k = i * 4000
    k = i * 300
    axes[0, i].set_title(f'{k}: {keys_reversed[k]}')
    axes[0, i].imshow(Image.open(folder / keys_reversed[k]))
    axes[1, i].plot(*dataset[k]['obs'].T)
    axes[1, i].axis('equal')

# Edit the zarr files to include the CLIP embeddings

In [ ]:
dataset_path_drawing = "data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr"
dataset_path_stroke = "data/gml_by_stroke_PRESERVE_ASPECT_CENTERED_003000.zarr"

key = 'rendering_clip'

In [ ]:
import zarr

# Open the Zarr dataset in a context manager
with zarr.open(dataset_path_drawing, mode='a') as dataset_root:
    if 'filenames' in dataset_root['meta']:
        print("Warning: 'filenames' already exists in 'meta'. It won't be overwritten.")
    else:
        n = len(dataset_root['meta']['episode_ends'])
        print(keys_reversed[:n])
        dataset_root['meta']['filenames'] = keys_reversed[:n]
        print(dataset_root['meta']['episode_ends'].shape)
        print('added filenames')
    if key in dataset_root['meta']:
        print(f"Warning: {key} already exists in 'meta'. It won't be overwritten.")
    else:
        filenames = dataset_root['meta']['filenames']
        dataset_root['meta'][key] = np.concatenate([all_features[k] for k in filenames])
        print('added CLIP embeddings')

    print(dataset_root['meta'][key][:].shape)
    print(dataset_root['meta'][key][:])

In [ ]:
# Do it for the stroke dataset
with zarr.open(dataset_path_drawing, mode='r') as dataset_root:
    drawing_episode_ends = dataset_root['meta']['episode_ends']

with zarr.open(dataset_path_stroke, mode='a') as dataset_root:
    stroke_episode_ends = dataset_root['meta']['episode_ends']
    stroke2drawing = np.zeros(len(dataset_root['meta']['episode_ends']), dtype=int)
    for i in reversed(range(len(drawing_episode_ends))):
        d_end = drawing_episode_ends[i]
        stroke2drawing[stroke_episode_ends <= d_end] = i
    print(stroke2drawing)
    print(stroke2drawing[:50])
    print(stroke2drawing[-20:])


    if 'filenames' in dataset_root['meta']:
        print("Warning: 'filenames' already exists in 'meta'. It won't be overwritten.")
    else:
        n = len(drawing_episode_ends)
        print(keys_reversed[:n])
        dataset_root['meta']['filenames'] = [keys_reversed[i] for i in stroke2drawing]
        print(dataset_root['meta']['episode_ends'].shape)
        print('added filenames')
    if key in dataset_root['meta']:
        print(f"Warning: {key} already exists in 'meta'. It won't be overwritten.")
    else:
        filenames = dataset_root['meta']['filenames']
        dataset_root['meta'][key] = np.concatenate([all_features[k] for k in filenames])
        print('added CLIP embeddings')

    print(dataset_root['meta'][key][:].shape)
    print(dataset_root['meta'][key][:])

In [ ]:
# Spot check CLIP embeddings

fig, axes = plt.subplots(2, 10, figsize=(20, 5))

# Drawing dataset
dataset = GmlDatasetNoSliding(dataset_path_drawing, 512, use_clip_embeddings='rendering')
for i, k in enumerate(range(0, 3000, 300)):
    sample = dataset[k]
    filename = sample['filename']

    im_orig = Image.open(folder / filename)
    axes[0, i].set_title(f'{k}: {filename}')
    axes[0, i].imshow(im_orig)
    axes[1, i].plot(*sample['obs'].T)
    axes[1, i].axis('equal')

    image = preprocess(im_orig).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image).cpu().numpy()
    
    assert np.allclose(image_features, sample['clip']), f"CLIP embeddings don't match for {k}"

In [ ]:
# Confusion matrix
features = np.array([dataset[k]['clip'] for k in range(0, 3000, 300)])
features.shape
similarity_matrix_1 = np.einsum('ij,kj->ik', features, features)
similarity_matrix = np.einsum('ij,kj->ik', features, features) / np.linalg.norm(features, axis=1) / np.linalg.norm(features, axis=1)[:, None]
with np.printoptions(precision=3, suppress=True):
    # print(similarity_matrix_1)
    print(similarity_matrix)
# plt.imshow(similarity_matrix_1)
plt.imshow(similarity_matrix)

In [ ]:
# Spot check CLIP embeddings, STROKE dataset

fig, axes = plt.subplots(2, 10, figsize=(20, 5))

dataset = GmlDatasetNoSliding(dataset_path_stroke, 512, use_clip_embeddings='rendering')
indices = np.linspace(0, len(dataset) - 1, 10, dtype=int)
for i, k in enumerate(indices):
    print(k, dataset[k]['filename'])
    sample = dataset[k]
    filename = sample['filename']
    # find all with the same filename
    idx = []
    for j in range(k - 50, k + 50):
        if j < 0 or j >= len(dataset):
            continue
        if dataset[j]['filename'] == filename:
            idx.append(j)

    im_orig = Image.open(folder / filename)
    axes[0, i].set_title(f'{k}: {filename}')
    axes[0, i].imshow(im_orig)
    for j in idx:
        obs = dataset[j]['obs']
        obs[obs == 0] = np.nan
        axes[1, i].plot(*obs.T)
    axes[1, i].axis('equal')

    image = preprocess(im_orig).unsqueeze(0).to(device)
    with torch.no_grad():
        image_features = model.encode_image(image).cpu().numpy()
    
    assert np.allclose(image_features, sample['clip']), f"CLIP embeddings don't match for {k}"

# dataset = GmlDatasetNoSliding(dataset_path_stroke, 128)